# Notebook 01 — Entendimento e Qualidade dos Dados

Este notebook estabelece a leitura técnica das fontes do case, valida granularidade, chaves, cobertura e riscos de vazamento de informação. O objetivo não é esgotar análises descritivas, mas documentar as decisões necessárias para construir o target e a base analítica com consistência temporal.

**Entregas desta etapa**

- inventário das quatro bases e das principais chaves;
- validação da granularidade de contratos e parcelas;
- diagnóstico de ausências, tipos, relacionamentos e valores especiais;
- definição das informações indisponíveis no momento da decisão;
- direcionamento técnico para os Notebooks 02 e 03.

## 1. Configuração e carregamento

Os caminhos são parametrizados para execução em ambiente local e podem ser substituídos por variáveis de ambiente. O usuário pode definir `CREDIT_RISK_DATA_PATH` e `CREDIT_RISK_OUTPUT_PATH` ou preencher os caminhos manuais.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F

# Garante que driver e workers utilizem o Python do kernel ativo.
PYTHON_EXECUTAVEL = str(Path(sys.executable).resolve())

os.environ["PYSPARK_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_EXECUTAVEL
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

try:
    spark.stop()
except (NameError, AttributeError):
    pass

spark = (
    SparkSession.builder
    .master(os.getenv("SPARK_MASTER", "local[2]"))
    .appName("credit-risk-case")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", PYTHON_EXECUTAVEL)
    .config("spark.pyspark.driver.python", PYTHON_EXECUTAVEL)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Ajuste manual opcional. Na ausência de valor, são utilizadas variáveis de ambiente,
# um Volume padrão do Databricks, quando disponível, ou pastas locais relativas.
CAMINHO_DADOS_MANUAL = None
CAMINHO_SAIDA_MANUAL = None


def resolver_caminho(caminho_manual, variavel_ambiente, caminho_databricks, caminho_local):
    if caminho_manual:
        return str(caminho_manual)
    if os.getenv(variavel_ambiente):
        return os.environ[variavel_ambiente]
    if caminho_databricks and Path(caminho_databricks).exists():
        return caminho_databricks
    return caminho_local


def juntar_caminho(diretorio, arquivo):
    return f"{str(diretorio).rstrip('/')}/{arquivo}"


DATA_PATH = resolver_caminho(
    CAMINHO_DADOS_MANUAL,
    "CREDIT_RISK_DATA_PATH",
    "/Volumes/workspace/default/credit_risk_data",
    "data"
)

OUTPUT_PATH = resolver_caminho(
    CAMINHO_SAIDA_MANUAL,
    "CREDIT_RISK_OUTPUT_PATH",
    "/Volumes/workspace/default/case_tecnico_ds",
    "outputs"
)

if "://" not in OUTPUT_PATH:
    Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)


def visualizar(df, n=20, truncate=False):
    if n <= 0:
        raise ValueError("O parâmetro 'n' deve ser maior que zero.")
    df.show(n=n, truncate=truncate)


print(f"Python: {sys.version.split()[0]} | Spark: {spark.version}")
print(f"Dados: {DATA_PATH}")
print(f"Saídas: {OUTPUT_PATH}")

Python: 3.11.7 | Spark: 4.1.2
Dados: data
Saídas: outputs


In [2]:
base_cadastral = spark.read.parquet(juntar_caminho(DATA_PATH, "base_cadastral.parquet"))
base_submissao = spark.read.parquet(juntar_caminho(DATA_PATH, "base_submissao.parquet"))
historico_emprestimos = spark.read.parquet(juntar_caminho(DATA_PATH, "historico_emprestimos.parquet"))
historico_parcelas = spark.read.parquet(juntar_caminho(DATA_PATH, "historico_parcelas.parquet"))

bases = {
    "base_cadastral": base_cadastral,
    "base_submissao": base_submissao,
    "historico_emprestimos": historico_emprestimos,
    "historico_parcelas": historico_parcelas
}

## 2. Estrutura e inventário das bases

A primeira validação confirma volume, quantidade de colunas e aderência do schema ao dicionário. A leitura do dicionário é auxiliar: a execução principal não depende dele.

In [3]:
contagem_bases = {nome: df.count() for nome, df in bases.items()}

dimensoes = spark.createDataFrame([
    Row(tabela=nome, linhas=contagem_bases[nome], colunas=len(df.columns))
    for nome, df in bases.items()
])
visualizar(dimensoes.orderBy("linhas"))

inventario_tecnico = spark.createDataFrame([
    Row(base=nome, coluna=campo.name, tipo_spark=campo.dataType.simpleString())
    for nome, df in bases.items()
    for campo in df.schema.fields
])

caminhos_dicionario = [
    juntar_caminho(DATA_PATH, "dicionario_dados.csv"),
    juntar_caminho(DATA_PATH, "dicionario_dados.xlsx")
]

caminho_dicionario = next(
    (caminho for caminho in caminhos_dicionario if Path(caminho).exists()),
    None
)

if caminho_dicionario:
    if caminho_dicionario.endswith(".csv"):
        try:
            dicionario_pandas = pd.read_csv(
                caminho_dicionario,
                sep=None,
                engine="python",
                encoding="utf-8"
            )
        except UnicodeDecodeError:
            dicionario_pandas = pd.read_csv(
                caminho_dicionario,
                sep=None,
                engine="python",
                encoding="latin1"
            )
    else:
        dicionario_pandas = pd.read_excel(
            caminho_dicionario
        )

    def corrigir_codificacao(valor):
        if not isinstance(valor, str) or "Ã" not in valor:
            return valor

        try:
            return valor.encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return valor

    for coluna in dicionario_pandas.select_dtypes(
        include="object"
    ).columns:
        dicionario_pandas[coluna] = (
            dicionario_pandas[coluna]
            .map(corrigir_codificacao)
        )

    dicionario_dados = spark.createDataFrame(
        dicionario_pandas
    )

    dicionario_variaveis = (
        inventario_tecnico
        .join(
            dicionario_dados,
            ["base", "coluna"],
            "left"
        )
        .select(
            "base",
            "coluna",
            "descricao",
            "tipo_spark"
        )
        .orderBy(
            "base",
            "coluna"
        )
    )

    visualizar(
        dicionario_variaveis,
        n=200
    )

else:
    print(
        "Dicionário não encontrado; "
        "o pipeline principal permanece executável."
    )

    visualizar(
        inventario_tecnico.orderBy(
            "base",
            "coluna"
        ),
        n=200
    )


+---------------------+-------+-------+
|tabela               |linhas |colunas|
+---------------------+-------+-------+
|base_cadastral       |40000  |16     |
|base_submissao       |40000  |8      |
|historico_emprestimos|186890 |37     |
|historico_parcelas   |1390978|8      |
+---------------------+-------+-------+

+---------------------+--------------------------------+-----------------------------------------------------------------------+----------+
|base                 |coluna                          |descricao                                                              |tipo_spark|
+---------------------+--------------------------------+-----------------------------------------------------------------------+----------+
|base_cadastral       |data_nascimento                 |Data de nascimento do cliente                                          |string    |
|base_cadastral       |estado_civil                    |Estado civil do cliente                                        

## 3. Chaves, cobertura e granularidade

A base cadastral e a base de submissão devem possuir uma linha por cliente; o histórico de empréstimos, uma linha por contrato. Em `historico_parcelas`, a análise deve distinguir parcela econômica de eventos e versões de pagamento.

In [4]:
def validar_unicidade(df, nome_tabela, colunas_chave, total_linhas):
    distintos = df.select(*colunas_chave).distinct().count()
    return Row(
        tabela=nome_tabela,
        chave=" + ".join(colunas_chave),
        total_linhas=total_linhas,
        chaves_distintas=distintos,
        chave_unica="Sim" if total_linhas == distintos else "Não"
    )

validacoes_chaves = spark.createDataFrame([
    validar_unicidade(
        base_cadastral, "base_cadastral", ["id_cliente"], contagem_bases["base_cadastral"]
    ),
    validar_unicidade(
        base_submissao, "base_submissao", ["id_cliente"], contagem_bases["base_submissao"]
    ),
    validar_unicidade(
        historico_emprestimos, "historico_emprestimos", ["id_contrato"],
        contagem_bases["historico_emprestimos"]
    )
])
visualizar(validacoes_chaves)

cobertura_submissao = (
    base_submissao.select("id_cliente").alias("sub")
    .join(
        base_cadastral.select("id_cliente").alias("cad"),
        "id_cliente",
        "left"
    )
    .agg(
        F.lit(contagem_bases["base_cadastral"]).alias("clientes_cadastral"),
        F.count("*").alias("clientes_submissao"),
        F.sum(F.col("cad.id_cliente").isNotNull().cast("int")).alias("clientes_em_comum")
    )
)
visualizar(cobertura_submissao)

+---------------------+-----------+------------+----------------+-----------+
|tabela               |chave      |total_linhas|chaves_distintas|chave_unica|
+---------------------+-----------+------------+----------------+-----------+
|base_cadastral       |id_cliente |40000       |40000           |Sim        |
|base_submissao       |id_cliente |40000       |40000           |Sim        |
|historico_emprestimos|id_contrato|186890      |186890          |Sim        |
+---------------------+-----------+------------+----------------+-----------+

+------------------+------------------+-----------------+
|clientes_cadastral|clientes_submissao|clientes_em_comum|
+------------------+------------------+-----------------+
|40000             |40000             |40000            |
+------------------+------------------+-----------------+



In [5]:
total_linhas_parcelas = contagem_bases["historico_parcelas"]
linhas_distintas_parcelas = historico_parcelas.dropDuplicates().count()

CHAVES_CANDIDATAS_PARCELAS = [
    ["id_contrato"],
    ["id_contrato", "numero_parcela"],
    ["id_contrato", "numero_parcela", "versao_parcela"]
]

validacao_granularidade_parcelas = spark.createDataFrame([
    validar_unicidade(
        historico_parcelas,
        "historico_parcelas",
        chave,
        total_linhas_parcelas
    )
    for chave in CHAVES_CANDIDATAS_PARCELAS
])

grupos_repetidos_parcela_versao = (
    historico_parcelas
    .groupBy("id_contrato", "numero_parcela", "versao_parcela")
    .count()
    .filter(F.col("count") > 1)
)

resumo_granularidade = grupos_repetidos_parcela_versao.agg(
    F.lit(total_linhas_parcelas).alias("total_linhas"),
    F.lit(total_linhas_parcelas - linhas_distintas_parcelas).alias("duplicidades_exatas"),
    F.count("*").alias("grupos_parcela_versao_repetidos")
)

visualizar(validacao_granularidade_parcelas)
visualizar(resumo_granularidade)
visualizar(
    historico_parcelas.join(
        grupos_repetidos_parcela_versao.select(
            "id_contrato", "numero_parcela", "versao_parcela"
        ),
        ["id_contrato", "numero_parcela", "versao_parcela"],
        "inner"
    ).orderBy("id_contrato", "numero_parcela", "versao_parcela")
)

+------------------+---------------------------------------------+------------+----------------+-----------+
|tabela            |chave                                        |total_linhas|chaves_distintas|chave_unica|
+------------------+---------------------------------------------+------------+----------------+-----------+
|historico_parcelas|id_contrato                                  |1390978     |107419          |Não        |
|historico_parcelas|id_contrato + numero_parcela                 |1390978     |1310932         |Não        |
|historico_parcelas|id_contrato + numero_parcela + versao_parcela|1390978     |1320960         |Não        |
+------------------+---------------------------------------------+------------+----------------+-----------+

+------------+-------------------+-------------------------------+
|total_linhas|duplicidades_exatas|grupos_parcela_versao_repetidos|
+------------+-------------------+-------------------------------+
|1390978     |0                  |6

**Decisão de granularidade**

`historico_parcelas` não representa necessariamente uma única linha por parcela econômica. Há versões e múltiplos eventos de pagamento para a mesma combinação de contrato, número e versão. Portanto, somar diretamente os registros duplicaria valores previstos e pagos. O Notebook 02 consolidará primeiro as versões e os eventos antes de calcular atraso e target.

**Mapa de relacionamentos**

| Origem | Destino | Chave | Cardinalidade esperada |
|---|---|---|---|
| `base_cadastral` | `base_submissao` | `id_cliente` | 1:1 |
| `base_cadastral` | `historico_emprestimos` | `id_cliente` | 1:N |
| `historico_emprestimos` | `historico_parcelas` | `id_contrato` + `id_cliente` | 1:N |

## 4. Qualidade e consistência dos dados

Os diagnósticos seguintes separam três situações: ausência real de cadastro, ausência estrutural por inexistência de histórico e inconsistência que exige tratamento. Essa distinção evita imputações genéricas sem interpretação de negócio.

In [6]:
resultado_nulos = []

for nome_base, df in bases.items():
    total = contagem_bases[nome_base]
    contagens = df.agg(*[
        F.sum(F.col(coluna).isNull().cast("int")).alias(coluna)
        for coluna in df.columns
    ]).first().asDict()

    resultado_nulos.extend([
        Row(
            tabela=nome_base,
            variavel=coluna,
            qtd_nulos=qtd,
            perc_nulos=round(qtd / total * 100, 4)
        )
        for coluna, qtd in contagens.items()
        if qtd > 0
    ])

nulos_df = spark.createDataFrame(resultado_nulos)
visualizar(nulos_df.orderBy(F.desc("perc_nulos")), n=200)

+---------------------+-------------------------------+---------+----------+
|tabela               |variavel                       |qtd_nulos|perc_nulos|
+---------------------+-------------------------------+---------+----------+
|historico_emprestimos|taxa_juros_padrao              |186262   |99.664    |
|historico_emprestimos|taxa_juros_promocional         |186262   |99.664    |
|historico_emprestimos|data_liberacao                 |179790   |96.201    |
|historico_emprestimos|data_encerramento              |100803   |53.9371   |
|historico_emprestimos|valor_entrada                  |100168   |53.5973   |
|historico_emprestimos|percentual_entrada             |100168   |53.5973   |
|historico_emprestimos|data_ultimo_vencimento         |99112    |53.0323   |
|historico_emprestimos|acompanhantes_cliente          |92104    |49.2825   |
|historico_emprestimos|data_ultimo_vencimento_original|85747    |45.881    |
|historico_emprestimos|data_primeiro_vencimento       |79745    |42.6695   |

In [7]:
resultado_datas = []

for nome_base, df in bases.items():
    colunas_data = [c for c in df.columns if "data" in c.lower()]
    if not colunas_data:
        continue

    contagens_invalidas = df.agg(*[
        F.sum(
            F.when(
                F.col(coluna).isNotNull()
                & F.to_date(F.col(coluna), "yyyy-MM-dd").isNull(),
                1
            ).otherwise(0)
        ).alias(coluna)
        for coluna in colunas_data
    ]).first().asDict()

    tipos = dict(df.dtypes)
    resultado_datas.extend([
        Row(
            tabela=nome_base,
            variavel=coluna,
            tipo_atual=tipos[coluna],
            qtd_datas_invalidas=contagens_invalidas[coluna]
        )
        for coluna in colunas_data
    ])

validacao_datas_df = spark.createDataFrame(resultado_datas)
visualizar(
    validacao_datas_df.orderBy(F.desc("qtd_datas_invalidas"), "tabela", "variavel"),
    n=100
)

+---------------------+-------------------------------+----------+-------------------+
|tabela               |variavel                       |tipo_atual|qtd_datas_invalidas|
+---------------------+-------------------------------+----------+-------------------+
|base_cadastral       |data_nascimento                |string    |0                  |
|base_submissao       |data_solicitacao               |string    |0                  |
|historico_emprestimos|data_decisao                   |string    |0                  |
|historico_emprestimos|data_encerramento              |string    |0                  |
|historico_emprestimos|data_liberacao                 |string    |0                  |
|historico_emprestimos|data_primeiro_vencimento       |string    |0                  |
|historico_emprestimos|data_ultimo_vencimento         |string    |0                  |
|historico_emprestimos|data_ultimo_vencimento_original|string    |0                  |
|historico_parcelas   |data_prevista_pagame

In [8]:
contratos_orfaos = (
    historico_parcelas.select("id_contrato").distinct()
    .join(historico_emprestimos.select("id_contrato").distinct(), "id_contrato", "left_anti")
)

clientes_orfaos = (
    historico_emprestimos.select("id_cliente").distinct()
    .join(base_cadastral.select("id_cliente").distinct(), "id_cliente", "left_anti")
)

contratos_com_parcelas = (
    historico_parcelas.select("id_contrato").distinct()
    .withColumn("possui_historico_parcelas", F.lit(1))
)

aprovados_sem_parcelas = (
    historico_emprestimos
    .join(contratos_com_parcelas, "id_contrato", "left")
    .fillna({"possui_historico_parcelas": 0})
    .filter(
        (F.col("status_contrato") == "Approved")
        & (F.col("possui_historico_parcelas") == 0)
    )
)

resumo_integridade = (
    contratos_orfaos.agg(F.count("*").alias("contratos_parcelas_orfaos"))
    .crossJoin(
        clientes_orfaos.agg(F.count("*").alias("clientes_historico_sem_cadastro"))
    )
    .crossJoin(
        aprovados_sem_parcelas.agg(F.count("*").alias("aprovados_sem_parcelas"))
    )
)
visualizar(resumo_integridade)

visualizar(
    aprovados_sem_parcelas.agg(
        F.sum(F.col("data_liberacao").isNull().cast("int")).alias("sem_data_liberacao"),
        F.sum(F.col("data_primeiro_vencimento").isNull().cast("int")).alias("sem_primeiro_vencimento"),
        F.sum(
            (F.col("valor_credito").isNull() | (F.col("valor_credito") <= 0)).cast("int")
        ).alias("sem_valor_credito"),
        F.min(F.to_date("data_decisao")).alias("menor_data_decisao"),
        F.max(F.to_date("data_decisao")).alias("maior_data_decisao")
    )
)

+-------------------------+-------------------------------+----------------------+
|contratos_parcelas_orfaos|clientes_historico_sem_cadastro|aprovados_sem_parcelas|
+-------------------------+-------------------------------+----------------------+
|0                        |0                              |8763                  |
+-------------------------+-------------------------------+----------------------+

+------------------+-----------------------+-----------------+------------------+------------------+
|sem_data_liberacao|sem_primeiro_vencimento|sem_valor_credito|menor_data_decisao|maior_data_decisao|
+------------------+-----------------------+-----------------+------------------+------------------+
|8701              |8763                   |51               |2017-02-06        |2025-02-22        |
+------------------+-----------------------+-----------------+------------------+------------------+



**Decisão sobre contratos aprovados sem parcelas**

Contratos aprovados sem histórico de parcelas não possuem performance observável e não devem receber target de forma artificial. Eles podem ser analisados como parte do funil de concessão, mas ficam fora da população de desenvolvimento supervisionado.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

variaveis_numericas = {
    "base_cadastral": ["renda_anual", "qtd_filhos", "qtd_membros_familia"],
    "base_submissao": ["valor_credito", "valor_bem", "valor_parcela"],
    "historico_emprestimos": [
        "valor_credito", "valor_solicitado", "valor_entrada",
        "valor_parcela", "qtd_parcelas_planejadas"
    ],
    "historico_parcelas": ["valor_previsto_parcela", "valor_pago_parcela"]
}

schema_resultado_numericas = StructType([
    StructField("tabela", StringType(), False),
    StructField("variavel", StringType(), False),
    StructField("minimo", DoubleType(), True),
    StructField("mediana", DoubleType(), True),
    StructField("media", DoubleType(), True),
    StructField("maximo", DoubleType(), True)
])

resultado_numericas = []

for nome_base, colunas in variaveis_numericas.items():
    df = bases[nome_base]
    colunas_validas = [c for c in colunas if c in df.columns]

    expressoes = []
    for coluna in colunas_validas:
        valor = F.col(coluna).cast("double")
        expressoes.extend([
            F.min(valor).alias(f"{coluna}__minimo"),
            F.percentile_approx(valor, 0.5).cast("double").alias(f"{coluna}__mediana"),
            F.avg(valor).alias(f"{coluna}__media"),
            F.max(valor).alias(f"{coluna}__maximo")
        ])

    estatisticas = df.agg(*expressoes).first().asDict()

    for coluna in colunas_validas:
        minimo = estatisticas[f"{coluna}__minimo"]
        mediana = estatisticas[f"{coluna}__mediana"]
        media = estatisticas[f"{coluna}__media"]
        maximo = estatisticas[f"{coluna}__maximo"]
        resultado_numericas.append((
            nome_base,
            coluna,
            float(minimo) if minimo is not None else None,
            float(mediana) if mediana is not None else None,
            round(float(media), 2) if media is not None else None,
            float(maximo) if maximo is not None else None
        ))

resultado_numericas_df = spark.createDataFrame(
    resultado_numericas,
    schema=schema_resultado_numericas
)
visualizar(resultado_numericas_df.orderBy("tabela", "variavel"), n=100)

+---------------------+-----------------------+-------+--------+---------+----------+
|tabela               |variavel               |minimo |mediana |media    |maximo    |
+---------------------+-----------------------+-------+--------+---------+----------+
|base_cadastral       |qtd_filhos             |0.0    |0.0     |0.41     |14.0      |
|base_cadastral       |qtd_membros_familia    |1.0    |2.0     |2.15     |15.0      |
|base_cadastral       |renda_anual            |26100.0|148500.0|172853.87|1.17E8    |
|base_submissao       |valor_bem              |40500.0|450000.0|526968.2 |4050000.0 |
|base_submissao       |valor_credito          |45000.0|502186.5|586259.67|4050000.0 |
|base_submissao       |valor_parcela          |2295.0 |25056.0 |27376.03 |225000.0  |
|historico_emprestimos|qtd_parcelas_planejadas|0.0    |12.0    |16.02    |84.0      |
|historico_emprestimos|valor_credito          |0.0    |80100.0 |194709.44|3749053.5 |
|historico_emprestimos|valor_entrada          |0.0    

In [10]:
def distribuicao_categorica(df, coluna, total_linhas):
    return (
        df.groupBy(coluna)
        .count()
        .withColumn("percentual", F.round(F.col("count") / F.lit(total_linhas) * 100, 4))
        .orderBy(F.desc("count"))
    )

visualizar(
    distribuicao_categorica(
        historico_emprestimos, "status_contrato", contagem_bases["historico_emprestimos"]
    )
)
visualizar(
    distribuicao_categorica(
        historico_emprestimos, "tipo_produto", contagem_bases["historico_emprestimos"]
    )
)
visualizar(
    distribuicao_categorica(
        historico_emprestimos, "categoria_bem", contagem_bases["historico_emprestimos"]
    )
)
visualizar(
    distribuicao_categorica(
        base_cadastral, "tipo_renda", contagem_bases["base_cadastral"]
    )
)

+---------------+------+----------+
|status_contrato|count |percentual|
+---------------+------+----------+
|Approved       |116182|62.166    |
|Canceled       |35767 |19.138    |
|Refused        |32108 |17.1802   |
|Unused offer   |2833  |1.5159    |
+---------------+------+----------+

+------------+------+----------+
|tipo_produto|count |percentual|
+------------+------+----------+
|XNA         |119382|63.8782   |
|x-sell      |50625 |27.0881   |
|walk-in     |16883 |9.0337    |
+------------+------+----------+

+------------------------+------+----------+
|categoria_bem           |count |percentual|
+------------------------+------+----------+
|XNA                     |106340|56.8998   |
|Mobile                  |25122 |13.4421   |
|Consumer Electronics    |13760 |7.3626    |
|Computers               |11710 |6.2657    |
|Audio/Video             |11060 |5.9179    |
|Furniture               |6004  |3.2126    |
|Construction Materials  |2851  |1.5255    |
|Photo / Cinema Equipment|280

## 5. Riscos de leakage e disponibilidade temporal

A regra central é: uma feature só pode utilizar informação conhecida até o instante da avaliação. Variáveis posteriores à decisão do próprio contrato não entram no modelo, mesmo quando apresentam elevado poder explicativo.

| Grupo | Exemplos | Decisão |
|---|---|---|
| Resultado da decisão | `status_contrato`, motivo de recusa | excluir do contrato avaliado |
| Performance posterior | pagamentos, atraso, encerramento | usar apenas em contratos históricos anteriores ao corte |
| Datas pós-concessão | liberação, vencimento e encerramento | excluir do contrato avaliado; permitir somente no histórico |
| Dados disponíveis na solicitação | cadastro, valor solicitado, canal, produto | candidatos, após validação de qualidade |
| Identificadores | cliente e contrato | manter para rastreabilidade; nunca como preditores |

A engenharia de features do Notebook 03 adotará relacionamento point-in-time: para cada avaliação, somente contratos e pagamentos anteriores à respectiva `data_corte_features` poderão contribuir para as variáveis históricas.

## 6. Conclusões

- As quatro fontes possuem chaves suficientes para relacionar clientes, contratos e pagamentos.
- A submissão está integralmente coberta pela base cadastral.
- `historico_parcelas` contém versões e múltiplos eventos; sua granularidade exige consolidação antes do cálculo de atraso.
- Contratos aprovados sem parcelas não possuem resultado observável e ficam fora da construção do target.
- Datas devem ser convertidas explicitamente antes de comparações temporais.
- Ausências e valores especiais serão tratados conforme significado, e não por uma regra única.
- Informações posteriores à avaliação do contrato são bloqueadas para prevenir leakage.

## 7. Próximos passos

1. Consolidar parcelas e comparar definições de inadimplência no Notebook 02.
2. Construir a base analítica point-in-time para desenvolvimento e scoring no Notebook 03.
3. Treinar, validar e calibrar modelos; propor política de crédito e gerar a submissão no Notebook 04.